<a id="unit-u.810bec7753bd4fd7880b19e5ef9bc039"></a>

<div>
  <div>5-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Пакеттік жүктеу, сәтсіздіктер және қайта әрекеттер</div>
</div>

Он модельденген жаңа тапсырыс CSV файлы түрінде келеді. Оларды әр жолын оқуға болатын Stream Load сұрауымен жүктеп, жауап пен кестені тексересіз, пакетті сол label-мен қайта жібересіз және бір қате жолы бар пакеттің толық сәтсіз болуын көресіз. Содан кейін сол тапсырыстарды бір-бірден Group Commit-сіз және онымен жіберіп, әр тәсіл сақтауда не қалдыратынын салыстырасыз. Соңында WWI тарихының он кестесін Parquet-тен жүктеп, тексересіз. Шамамен 45 минут жоспарлаңыз.

**5-модуль: деректерді пакеттік және үздіксіз жүктеу** · [Сабақ](course.md) · [Тест](quiz05_load_methods_and_retry_safety.ipynb) · [Курстың README файлы](../../README.md)


<a id="unit-u.20fe6b1bd9d74fe382c16999069acb9d"></a>

<a id="lab-scope"></a>

## Зертханалық жұмыстың қамту аясы

Бұл жұмыс `orders_imported`, `orders_one_by_one`, `orders_group_commit` және он WWI тарихи кестесін — `wwi_orders`, `wwi_order_lines`, `wwi_customers`, `wwi_products`, `wwi_invoices`, `wwi_invoice_lines`, `wwi_customer_transactions`, `wwi_payment_methods`, `wwi_transaction_types`, `wwi_delivery_methods` — қайта жасайды. Жаттығу `orders_mapping_practice` кестесін қайта жасайды. 1-зертханалық жұмыстың курс сынақ ортасы қажет, 4-зертханалық жұмыстың деректер көлі ортасы қажет емес.

Модельденген CSV мен WWI Parquet пакеті репозиторийде; алғаш қолданғанда пакет ашылып, курс манифесімен салыстырылып тексеріледі. Оларды курстың README файлындағы [Деректер жиыны](../../README.md#dataset) бөлімі сипаттайды.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, expect
from dw_course.schema import ORDER_COLUMNS, order_ddl
from dw_course.ui import show_sql, show_response, show_records

lab = connect_sandbox()

<a id="unit-u.a37e5e69017e40b6975d047a41526139"></a>

<a id="1-load-ten-simulated-new-orders"></a>

## 1. Он модельденген жаңа тапсырысты жүктеу

900001–900010 тапсырыстары — жалпы сомасы 1400.00 болатын `COURSE_SIMULATION` тапсырыстары. Алғашқы жүктеу әр бөлікті оқу үшін толық HTTP сұрауын жазады:

1. URL BE HTTP портындағы зертханалық дерекқорыңыздың `orders_imported` кестесін атайды. Қосылу қадамы сол мекенжайды `DW_BE_HTTP_URL` ішіне қойған.
2. Тақырыптарда параметрлер бар: пішім, бөлгіш, файл ретіндегі `columns`, осы пакеттің **label** мәні, `strict_mode`, `max_filter_ratio` = 0 және `group_commit` = `off_mode`.
3. `requests.put` қосылым пайдаланушысы мен құпиясөзін қолданып, файл байттарын сұрау денесі ретінде жібереді.
4. `raise_for_status()` HTTP қатесінде тоқтайды, `response.json()` Doris жауабын оқиды, ал тексерулер оның сандарын, содан кейін кестені салыстырады.

Stream Load әдетте FE-ге барады, ол клиентті BE-ге бағыттайды. Бұл сынақ ортасында BE контейнер ішіндегі `127.0.0.1:8040` мекенжайын тіркеген, сондықтан қайта бағыттау клиентіңізді өз машинаңызда BE емес портқа жіберер еді. Сондықтан жұмыс сұрауды BE-нің жарияланған 51040 портына тікелей жіберіп, қайта бағыттаудан бас тартады. Label-ді сақтаңыз: 2-қадам пакетті сол арқылы қайта жібереді.


In [ ]:
import os
import requests

lab.execute("DROP TABLE IF EXISTS orders_imported")
ddl = order_ddl("orders_imported")
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
label = "dw_l1_" + uuid4().hex
columns = ",".join(ORDER_COLUMNS)
url = os.environ["DW_BE_HTTP_URL"] + f"/api/{lab.database}/orders_imported/_stream_load"
headers = {
    "label": label, "format": "csv", "column_separator": ",",
    "columns": columns, "strict_mode": "true", "max_filter_ratio": "0",
    "group_commit": "off_mode",
}
with (COURSE_ROOT / "datasets/orders.csv").open("rb") as payload:
    response = requests.put(url, auth=(lab.user, lab.password),
                            headers=headers, data=payload, timeout=120,
                            allow_redirects=False)
response.raise_for_status()
result = response.json()
show_response(result)
expect((result["Status"], result["NumberLoadedRows"], result["NumberFilteredRows"]), ("Success", 10, 0),
       title="The load reports ten rows loaded and none filtered")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")],
       title="The table holds ten orders, 1400.00")
lab.sql("SELECT order_id, customer_id, order_amount FROM orders_imported ORDER BY order_id", title="Ten loaded new orders");

<a id="unit-u.0becca6b7fd141a199153d61dca74fd2"></a>

**Күтілетін нәтиже**

Кесте анықтамасы, содан кейін өрістер кестесі түріндегі жауап: `Status` Success, `NumberTotalRows` 10, `NumberLoadedRows` 10, `NumberFilteredRows` 0, жіберген `Label` және жүктеу транзакциясы — `TxnId`. Екі жасыл белгі және 900001–900010 он тапсырысы: 900001 тапсырысы 1-тұтынушыға тиесілі, сомасы 100.00.


<a id="unit-u.ba93ab3fa17343ba9cf4150bf2e22dab"></a>

<a id="2-resend-the-same-batch-with-the-same-label"></a>

## 2. Сол пакетті сол label-мен қайта жіберу

1-қадам жауабы жоғалды деп елестетіп, сол файлды сол label-мен қайта жіберіңіз. `lab.stream_load()` 1-қадамдағыдай мекенжай, тіркелгі деректері, бағандар және параметрлермен сол сұрауды жібереді; бұдан кейін жұмыс осыны қолданады.

Doris аяқталған жүктеулердің label-дерін, Stream Load label-дерін 12 сағат есте сақтайды, сондықтан бекітілген пакетті қайталау ештеңе жүктемейді. Жаңа label жаңа пакет болар еді.


In [ ]:
retry = lab.stream_load("orders_imported", COURSE_ROOT / "datasets/orders.csv", label, columns)
show_response(retry)
expect(retry["Status"], "Label Already Exists", title="Doris recognizes the batch")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")],
       title="The table still holds ten orders, 1400.00")

<a id="unit-u.fe2f61050a374d5fae3a11ee0120a537"></a>

**Күтілетін нәтиже**

`Status` Label Already Exists және `ExistingJobStatus` FINISHED: алғашқы әрекет бекітілді, ал бұл әрекет ештеңе жүктемеді. Кестеде әлі он тапсырыс және 1400.00 бар.


<a id="unit-u.f9fe8d9efbce41daa7cff2c62a947e85"></a>

<a id="3-load-a-batch-with-one-bad-row"></a>

## 3. Бір қате жолы бар пакетті жүктеу

`malformed_orders.csv` файлын жаңа label-мен жүктеңіз: екі тапсырыс, екіншісінің сомасы — `not-a-number`. `strict_mode` және `max_filter_ratio` = 0 болса, бір қабылданбаған жол бүкіл пакетті сәтсіз етеді, сондықтан бірінші дұрыс жол да жүктелмейді.


In [ ]:
rejected = lab.stream_load("orders_imported", COURSE_ROOT / "datasets/malformed_orders.csv",
                           "dw_bad_" + uuid4().hex, columns)
show_response(rejected)
expect((rejected["Status"], rejected["NumberFilteredRows"]), ("Fail", 1), title="The batch fails because of one row")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")],
       title="The table is unchanged")
# ErrorURL names the BE's address inside the container; read the same path from the host's BE port.
error_url = rejected["ErrorURL"]
print(requests.get(os.environ["DW_BE_HTTP_URL"] + error_url[error_url.index("/api/"):], timeout=10).text)
lab.sql("SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_imported", title="Original data remains intact after rejection of the invalid batch");

<a id="unit-u.244e3544ad8c471f8f1816c0022ba332"></a>

**Күтілетін нәтиже**

`Status` Fail, `Message` `[DATA_QUALITY_ERROR]too many filtered rows`, `NumberTotalRows` 2, `NumberLoadedRows` 0, `NumberFilteredRows` 1 және `ErrorURL`. Бұл URL контейнер ішіндегі BE мекенжайын, `127.0.0.1:8040`, атайды; ұяшық сол жолды хосттың BE портынан оқып, қабылданбаған жолмен бірге себебін шығарады: `Reason: column(order_amount) value is incorrect while strict mode is true, src value is not-a-number`. Doris мұндай қате журналдарын біраз уақыт қана сақтайды: олар тұрақты жазба емес, ақауды анықтау белгілері. Кестеде әлі он тапсырыс және 1400.00 бар.


<a id="unit-u.3cade2d5eeb24c3484f0f06f271bf53f"></a>

<a id="4-many-small-loads-without-and-with-group-commit"></a>

## 4. Group Commit-сіз және онымен көп шағын жүктеу

Қолданба жиі бір тапсырыстан жібереді. Он тапсырысты бір жолдан он Stream Load ретінде екі рет жіберіңіз: `orders_one_by_one` ішіне `group_commit` = `off_mode` арқылы, әрқайсысын өз label-імен және `orders_group_commit` ішіне `async_mode` арқылы, label-сіз, себебі Group Commit жүктеуінде label болмауы керек. Екі кестеге де `group_commit_interval_ms` = 2000 беріледі, сондықтан топ әдепкі 10 секундтың орнына 2 секундтан кейін бекітіледі.

Әр кесте үшін ұяшық он жауаптың транзакция ID-лерін, оныншы жауаптан кейін бірден сұрау тапқан жолдар санын, он жолдың бәрі көрінгенше өткен уақытты және tablet нұсқасын санайды.


In [ ]:
import time

def load_one_line(table, line, mode):
    """Send one CSV line as its own Stream Load; a Group Commit load may not carry a label."""
    headers = {"format": "csv", "column_separator": ",", "columns": columns,
               "strict_mode": "true", "max_filter_ratio": "0", "group_commit": mode}
    if mode == "off_mode":
        headers["label"] = f"{table}_{uuid4().hex}"
    response = requests.put(f"{os.environ['DW_BE_HTTP_URL']}/api/{lab.database}/{table}/_stream_load",
                            auth=(lab.user, lab.password), headers=headers, data=(line + "\n").encode(),
                            timeout=120, allow_redirects=False)
    response.raise_for_status()
    return response.json()

lines = (COURSE_ROOT / "datasets/orders.csv").read_text().splitlines()
comparison = []
for table, mode in (("orders_one_by_one", "off_mode"), ("orders_group_commit", "async_mode")):
    lab.execute(f"DROP TABLE IF EXISTS {table}")
    lab.execute(order_ddl(table))
    interval = f'ALTER TABLE {table} SET ("group_commit_interval_ms" = "2000")'
    show_sql(f"{table}: commit a group after 2 seconds", interval)
    lab.execute(interval)
    responses = [load_one_line(table, line, mode) for line in lines]
    sent = time.monotonic()
    expect([response["Status"] for response in responses], ["Success"] * 10, title=f"{mode}: ten loads succeed")
    visible_at_once = lab.query(f"SELECT COUNT(*) FROM {table}")[0][0]
    while lab.query(f"SELECT COUNT(*) FROM {table}")[0][0] < 10:
        if time.monotonic() - sent > 60:
            raise TimeoutError(f"The rows of {table} were not visible after 60 seconds")
        time.sleep(0.2)
    comparison.append({
        "table": table, "group_commit": mode, "loads": len(responses),
        "transactions": len({response["TxnId"] for response in responses}),
        "visible right after the last answer": visible_at_once,
        "seconds until all 10 visible": round(time.monotonic() - sent, 1),
        "tablet version": lab.query(f"SHOW TABLETS FROM {table}")[0][4],
    })
show_response(responses[-1], title="The last async_mode response")
show_records("Ten single-row loads, without and with Group Commit", comparison)
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_group_commit"), [(10, "1400.00")],
       title="Group Commit loaded the same ten orders, 1400.00")

<a id="unit-u.2eaebdc7593b4321a2a2496835a2d622"></a>

**Күтілетін нәтиже**

Екі `ALTER TABLE` операторы, әр түрдің он жүктеуінің бәрі сәтті болғанын көрсететін екі жасыл белгі, соңғы `async_mode` жауабы, Group Commit сол он тапсырысты және 1400.00 жүктегенін көрсететін жасыл белгі және салыстыру:

| table | group_commit | transactions | visible right after the last answer | seconds until all 10 visible | tablet version |
| --- | --- | ---: | ---: | ---: | ---: |
| orders_one_by_one | off_mode | 10 | 10 | 0.0 | 11 |
| orders_group_commit | async_mode | 1 | 0 | шамамен 2 | 2 |

Group Commit-сіз он жүктеу — әрқайсысы жауап бергенде көрінетін он транзакция және кейін compaction біріктіруі керек он жаңа нұсқа. `async_mode` кезінде он жауап әр жол алдын ала жазу журналына түскен бойда келді, бәрінде бір транзакция ID-і бар, жолдар топ шамамен 2 секундтан кейін бекітілгенде бірге көрінді; tablet бір нұсқа алды. Сирек жағдайда он сұрау әлі келіп жатқанда топтың 2 секунды аяқталып, екі транзакция және 3-нұсқа болады. `async_mode` жауабында `GroupCommit` true және Doris жасаған `group_commit_…` түріндегі `Label` көрсетіледі. Соңғы тексеру сол он тапсырыс пен 1400.00 табады.


<a id="unit-u.7e905e2538d14ee48c05d9753e60415a"></a>

<a id="5-load-the-ten-tables-of-the-wwi-history"></a>

## 5. WWI тарихының он кестесін жүктеу

Parquet-тегі WWI тарихы үшін сол сұрау түрін қолданыңыз: он бизнес кестесі, барлығы 701,846 жол. Ұяшық алдымен әр файлды курс манифесімен тексереді, сондықтан жетіспейтін не өзгерген файл кез келген кесте жойылмай тұрып жұмысты тоқтатады. Әр кесте үшін манифестегі анықтаманы және жүктеу жауабын көрсетіп, жолдар санын және бастапқы кілттің бірегейлігін тексереді. Parquet файлы өрістерін атайды, сондықтан бұл жүктеулерге `columns` қажет емес.


In [ ]:
from dw_course.wwi import manifest, parquet_paths, parquet_ddl
paths = parquet_paths()  # Missing files or checksum mismatches stop execution before tables are dropped
for name, metadata in manifest()["tables"].items():
    target = "wwi_" + name
    lab.execute("DROP TABLE IF EXISTS " + target)
    ddl = parquet_ddl(name, target)
    show_sql("WWI historical table: " + name, ddl)
    lab.execute(ddl)
    response = lab.stream_load(target, paths[name], "wwi_" + uuid4().hex, format="parquet")
    show_response(response)
    expect((response["Status"], response["NumberLoadedRows"], response["NumberFilteredRows"]),
           ("Success", metadata["rows"], 0))
    primary = metadata["primary"]
    expect(lab.query(f"SELECT COUNT(*), COUNT(DISTINCT {primary}) FROM {target}"),
           [(metadata["rows"], metadata["rows"])], title=f"{target}: {metadata['rows']:,} rows, one per {primary}")

<a id="unit-u.50dda93fa9014daeb9edf5338a7b2119"></a>

**Күтілетін нәтиже**

Он анықтама, әрқайсысы сүзілген жолдары жоқ `Success` болатын он жауап және он жасыл белгі: `wwi_orders` 73,595 жол, `wwi_order_lines` 231,412, `wwi_customers` 663, `wwi_products` 227, `wwi_invoices` 70,510, `wwi_invoice_lines` 228,265, `wwi_customer_transactions` 97,147, `wwi_payment_methods` 4, `wwi_transaction_types` 13 және `wwi_delivery_methods` 10, әрқайсысының бастапқы кілті бірегей. Алғаш орындағанда пакет `.runtime/wwi/` ішіне ашылады, оған тағы бірнеше секунд кетеді.


<a id="unit-u.45cc8557512f492b9b7a238ab856c032"></a>

<a id="check-the-order-lines-and-their-relationships"></a>

### Тапсырыс жолдары мен байланыстарын тексеру

Әр кестені сәтті жүктеу кестелердің өзара сәйкес екенін дәлелдемейді. Тапсырыс жолдарын тапсырыстарымен, тұтынушыларымен және өнімдерімен біріктіріңіз: кілт жетіспесе не қайталанса, біріктіру жолдарды жоғалтар не көбейтер еді. Содан кейін тапсырыстарды күн бойынша санаңыз: `COUNT(DISTINCT o.OrderID)` тапсырыстарды санайды, себебі бір тапсырыстың бірнеше жолы бар. [Шот-фактуралар және шот түсімдері туралы қосымша оқу](optional05_invoice_and_receipts.md) тарихтың басқа сомаларын салыстырады; ол бұл жұмысқа кірмейді.


In [ ]:
expect(lab.query("""
SELECT COUNT(*), SUM(l.Quantity*l.UnitPrice)
FROM wwi_order_lines l
JOIN wwi_orders o ON l.OrderID=o.OrderID
JOIN wwi_customers c ON o.CustomerID=c.CustomerID
JOIN wwi_products p ON l.StockItemID=p.StockItemID
"""), [(231412, "177634276.40")], title="Every order line joins one order, one customer, and one product")
lab.sql("""
SELECT o.OrderDate, COUNT(DISTINCT o.OrderID) AS orders,
       SUM(l.Quantity*l.UnitPrice) AS order_amount
FROM wwi_orders o JOIN wwi_order_lines l ON o.OrderID=l.OrderID
GROUP BY o.OrderDate ORDER BY o.OrderDate LIMIT 10
""", title="Daily order analysis across the full history");

<a id="unit-u.f44797f83d5c42e381c28aff60edbe86"></a>

**Күтілетін нәтиже**

Жасыл белгі: барлық 231,412 тапсырыс жолының әрқайсысы бір тапсырысқа, бір тұтынушыға және бір өнімге бірігеді, олардың салыққа дейінгі жалпы сомасы — 177,634,276.40. Күндік есеп 2013-01-01 күнінен басталып, тарихтың алғашқы он күнін көрсетеді, әр күнде бірнеше ондағаннан шамамен жүзге дейін тапсырыс бар.


<a id="unit-u.80f7d4cfd2bf4aaa98bcb60b88bc0661"></a>

<a id="6-practice-map-columns-in-another-order"></a>

## 6. Жаттығу: бағандарды басқа ретпен сәйкестендіру

[orders_reordered.csv](../../datasets/orders_reordered.csv) екі жаңа тапсырысты қамтиды. Алғашқы екі бағаны — customer_id және order_id; қалғандары бастапқы CSV ретімен. orders_mapping_practice кестесін өзіңіз жасап, HTTP Stream Load арқылы дұрыс columns көрсетіңіз. 901001 тапсырысы 1-тұтынушыға тиесілі, сомасы 180, ал 901002 тапсырысы 2-тұтынушыға тиесілі, сомасы 80 екенін тексеріңіз.

1-қадамдағы HTTP мысалын жаңа мақсатты кесте, файл және label қолданып, columns ішінде алғашқы екі өрісті ауыстырып орындаңыз. Келесі ұяшықта ішінара толтырылған үлгі бар: түсініктеме белгілерін алып, баған сәйкестігі мен тексеруді толтырыңыз. Ол құпиясөзді ноутбукке жазбай, бар қосылым нысанынан оқиды. Бұл жаттығу тек orders_mapping_practice кестесін қайта жасайды.

Үлгі кестені жасағаннан кейін сол жүктеуді курс каталогындағы терминалдан да жіберуге болады. `<columns>` орнына өз сәйкестігіңізді қойыңыз; сынақ ортасы Stream Load-ты BE HTTP 51040 портында тікелей қабылдайды:

```bash
curl -u root: -T datasets/orders_reordered.csv \
  -H "label:mapping_curl_$(date +%s)" -H "format:csv" -H "column_separator:," \
  -H "columns:<columns>" \
  -H "strict_mode:true" -H "max_filter_ratio:0" -H "group_commit:off_mode" \
  http://127.0.0.1:51040/api/dw_course_l1_demo/orders_mapping_practice/_stream_load
```

Кодты келесі ұяшыққа жазыңыз, содан кейін одан кейінгі үлгі шешіммен салыстырыңыз.


In [ ]:
# # Select all lines in this cell, uncomment them (Ctrl+/ or Cmd+/), then replace each TODO.
# lab.execute("DROP TABLE IF EXISTS orders_mapping_practice")
# lab.execute(order_ddl("orders_mapping_practice"))
# url = os.environ["DW_BE_HTTP_URL"] + f"/api/{lab.database}/orders_mapping_practice/_stream_load"
# headers = {
#     "label": "mapping_" + uuid4().hex,
#     "format": "csv",
#     "column_separator": ",",
#     "columns": TODO_COLUMNS,  # TODO: the table column that each file column fills, in file order
#     "strict_mode": "true",
#     "max_filter_ratio": "0",
#     "group_commit": "off_mode",
# }
# with (COURSE_ROOT / "datasets/orders_reordered.csv").open("rb") as payload:
#     response = requests.put(url, auth=(lab.user, lab.password), headers=headers,
#                             data=payload, timeout=120, allow_redirects=False)
# response.raise_for_status()
# show_response(response.json(), title="Column-mapping load result")
# # TODO: query orders_mapping_practice and check the customer and amount of orders 901001 and 901002.


<a id="unit-u.9769e469e7de49189494a8b1ec0aa9a2"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
import os
import requests
lab.execute("DROP TABLE IF EXISTS orders_mapping_practice")
ddl = order_ddl("orders_mapping_practice")
show_sql("Exercise table structure", ddl)
lab.execute(ddl)
# The first two input columns are customer ID and order ID; the table uses a different field order.
practice_columns = "customer_id,order_id,order_amount,status,event_version,event_id,event_time,paid_amount,refund_amount,region,data_source"
practice_url = os.environ["DW_BE_HTTP_URL"] + f"/api/{lab.database}/orders_mapping_practice/_stream_load"
practice_headers = {
    "label": "mapping_" + uuid4().hex, "format": "csv", "column_separator": ",",
    "columns": practice_columns, "strict_mode": "true", "max_filter_ratio": "0",
    "group_commit": "off_mode",
}
with (COURSE_ROOT / "datasets/orders_reordered.csv").open("rb") as payload:
    response = requests.put(practice_url, auth=(lab.user, lab.password),
                            headers=practice_headers, data=payload, timeout=120,
                            allow_redirects=False)
response.raise_for_status()
result = response.json()
show_response(result, title="Independent column-mapping load result")
expect(result["Status"], "Success", title="The practice load succeeds")
lab.sql("SELECT order_id, customer_id, order_amount FROM orders_mapping_practice ORDER BY order_id", title="Order ID and customer ID mapping results");
expect(lab.query("SELECT order_id, customer_id, order_amount FROM orders_mapping_practice ORDER BY order_id"),
       [(901001,1,"180.00"),(901002,2,"80.00")], title="Each order has the right customer and amount")

<a id="unit-u.b78dc30cbd5d4ef1955cc54fc608e2da"></a>

**Күтілетін нәтиже**

Кесте анықтамасы, 2 жол жүктелген `Success` жауабы және екі тапсырыс: 1-тұтынушы үшін 180.00 сомалы 901001, 2-тұтынушы үшін 80.00 сомалы 901002. Ауыстырылған `columns` болмаса да, жүктеу сәтті болар еді, бірақ ID-лер ауысар еді: 901001 тұтынушыға тиесілі 1-тапсырыс.


<a id="unit-u.d0e180fbc55b429ab148ef4b2c2de411"></a>

<a id="7-practice-rank-the-products-by-sales"></a>

## 7. Жаттығу: өнімдерді сату бойынша реттеу

WWI өнімдерінің тапсырыс жолдары бойынша `SUM(Quantity * UnitPrice)` есептеп, салыққа дейінгі сатуы ең жоғары бес өнімді әр өнімнің жолдар санымен табыңыз. Содан кейін әр жолды дәл бір рет санағаныңызды тексеріңіз: барлық өнімнің сатуы барлық тапсырыс жолдарының жалпы сомасына тең болуы керек — 231,412 жол бойынша 177,634,276.40.


In [ ]:
# Your code here

<a id="unit-u.8bfbcd5835304a9386f12177c51a034d"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
top_products = """SELECT p.StockItemName, COUNT(*) AS order_lines,
       SUM(l.Quantity * l.UnitPrice) AS sales_amount
FROM wwi_order_lines l
JOIN wwi_products p ON l.StockItemID = p.StockItemID
GROUP BY p.StockItemName
ORDER BY sales_amount DESC
LIMIT 5"""
lab.sql(top_products, title="The five products with the highest sales");
expect(lab.query("""SELECT SUM(order_lines), SUM(sales_amount) FROM (
    SELECT COUNT(*) AS order_lines, SUM(l.Quantity * l.UnitPrice) AS sales_amount
    FROM wwi_order_lines l JOIN wwi_products p ON l.StockItemID = p.StockItemID
    GROUP BY p.StockItemID) per_product"""), [(231412, "177634276.40")],
       title="Every order line is counted once")

<a id="unit-u.9713c55bc37c47a6bfe747ac2ba36a10"></a>

**Күтілетін нәтиже**

Бес өнім, басында 1,061 тапсырыс жолы және 11,107,251.00 сомасы бар `Air cushion machine (Blue)`; келесі төрт көпіршікті орауыштың әрқайсысында шамамен 1,000–1,120 жол және 6.2–6.4 миллион бар. Тексеру жасыл: өнімдердің сатуы 231,412 жол бойынша 177,634,276.40 болады, сондықтан әр жол бір рет саналды. Тапсырыс жолдарын `wwi_orders` кестесіне де біріктіру әр жолды бір рет санар еді, себебі әр жолдың бір тапсырысы бар; шот-фактура жолдарын да біріктіру оларды көбейтер еді.


<a id="unit-u.f7ec6687711c417a84be5e367fb971a2"></a>

<a id="you-are-done-when"></a>

## Жұмысты аяқтау шарттары

- бір жүктеуден, сол label-мен қайталаудан және қабылданбаған пакеттен кейін `orders_imported` кестесінде он тапсырыс және 1400.00 бар;
- қайталау неге ештеңе жүктемегенін, бір қате жолы бар пакет неге ештеңе жүктемегенін және label қайталауды қанша уақыт қорғайтынын айта аласыз;
- Group Commit-сіз және онымен бір жолдан он жүктеудің транзакцияларын, нұсқаларын және жолдарының көріну уақытын салыстырдыңыз;
- он WWI кестесінде 701,846 жол бар, тапсырыс жолдары ештеңені жоғалтпай не екі еселемей бірігеді.

Жүктеу сәтсіз болса, жауабын сақтап, `ErrorURL` мәнін тезірек оқыңыз. `Publish Timeout` не жоғалған жауап кезінде қайталаудан бұрын бастапқы label нәтижесін тексеріңіз. 6-модуль қабылданбаған кірістерді кестеде сақтайды. Объектілік сақтаудан, Kafka-дан немесе дерекқор өзгерістерінен жүктеудің басқа тәсілдері сабақ пен қосымша оқу материалында берілген.


<a id="unit-u.464fda178e77474b815722879539c79a"></a>

<a id="next"></a>

## Келесі қадам

[5-тест: деректерді пакеттік және үздіксіз жүктеу](quiz05_load_methods_and_retry_safety.ipynb) файлын ашыңыз, содан кейін [6-модуль: деректер сапасы және схеманы тексеру](../module06-data-quality/course.md) бөліміне өтіңіз.
